In [57]:
import pandas as pd
import numpy as np
import os

In [58]:
from pathlib import Path
input_file_path = Path(r"D:\veena.sharma\My Documents\PROJECTS\PREDICTIVE_ANALYSIS\PSIPL Input Data")

print("Files available:")
for file in input_file_path.iterdir():
        print(f"  {file.name}")

Files available:
  Employee Attendance Report - PSIPL_20260826140835572.csv
  Employee Attendance Report_20260826125133276.csv
  Employee Attendance Report_20260826132240943.csv
  Employee Master PSIPL.xls
  FORM ONE F Report_20260807081614022.csv
  New Joiners.csv
  Overtime Upload Report_20260807084943902.csv
  Overtime Upload Report_20260807085253305.csv
  Overtime Upload Report_20260807085310437.csv
  Overtime Upload Report_20260807085324157.csv
  Resignation Report_20260807081605117.csv
  Yearly Leave Balance Report_20260806113614204.csv


In [59]:
employee_master = pd.read_excel(r"D:\veena.sharma\My Documents\PROJECTS\PREDICTIVE_ANALYSIS\PSIPL Input Data\Employee Master PSIPL.xls")
print("Shape:", employee_master.shape)
print("Columns:", employee_master.columns.tolist())

Shape: (20486, 92)
Columns: ['Group', 'Company Name', 'Employee Code', 'Title', 'First Name', 'Middle Name', 'Last Name', 'Employee Name', 'Gender', 'Nationality', 'Official Email Address', 'Personal Email Address', 'Resident Type', 'Marital Status', 'City Name', 'State Name', 'Branch', 'Site Branch', 'Division', 'Designation', 'Department', 'Sub Department', 'Cost Center', 'Site Cost Center', 'Site Name', 'Grade', 'Band', 'Reason for Requirement', 'Employee Category', 'Employee Status', 'Reporting Manager Code', 'Reporting Manager Name', 'Skip Level Manager Code', 'Skip Level Manager', 'Actual Join Date', 'Group Join Date', 'Resignation Date', 'Year Of Service (Current DOJ)', 'Year of Service (Group DOJ)', 'Physical Disability', 'Point of Hire', 'Point of Repatriation', 'ESIC Flag', 'ESIC Number', 'PF Applicable', 'PF Type', 'PF Number', 'UAN Number', 'FNF Settlement Date', 'Probation Status', 'Confirmation Date', 'Source of Hire', 'Preferred Name', 'Dual Nationality', 'Religion', 'Ag

In [60]:
required_cols = [
    "Employee Code",
    "Employee Status",
    "Actual Join Date",
    "Resignation Date",
    "Year Of Service (Current DOJ)",
    "Department",
    "Designation",
    "Company Name"
]

employee_master[required_cols].info()

print("\nMissing values:")
print(employee_master[required_cols].isnull().sum())

display(employee_master[required_cols].head(10))

<class 'pandas.DataFrame'>
RangeIndex: 20486 entries, 0 to 20485
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Employee Code                  20486 non-null  str           
 1   Employee Status                20486 non-null  str           
 2   Actual Join Date               20486 non-null  datetime64[us]
 3   Resignation Date               254 non-null    datetime64[us]
 4   Year Of Service (Current DOJ)  20486 non-null  int64         
 5   Department                     20486 non-null  str           
 6   Designation                    20486 non-null  str           
 7   Company Name                   20486 non-null  str           
dtypes: datetime64[us](2), int64(1), str(5)
memory usage: 1.3 MB

Missing values:
Employee Code                        0
Employee Status                      0
Actual Join Date                     0
Resignation Date                 2

,Employee Code,Employee Status,Actual Join Date,Resignation Date,Year Of Service (Current DOJ),Department,Designation,Company Name
0,'566137,Live,2025-04-20,NaT,1,Operations,Housekeeper,Property Solutions India Pvt Ltd
1,'534669,Live,2022-12-14,NaT,4,Operations,Office Executive,Property Solutions India Pvt Ltd
2,'560618,Live,2024-11-28,NaT,2,Operations,Bms Operator,Property Solutions India Pvt Ltd
3,'48681,Live,2014-04-20,NaT,12,Operations,Assistant Facility Manager,Property Solutions India Pvt Ltd
4,'62969,Live,2016-05-08,NaT,10,Operations,Steward,Property Solutions India Pvt Ltd
5,'62053,Live,2016-04-01,NaT,10,Operations,Electrician,Property Solutions India Pvt Ltd
6,'522934,Live,2021-12-16,NaT,5,Operations,Housekeeper,Property Solutions India Pvt Ltd
7,'550389,Live,2024-03-12,NaT,2,Operations,Office Assistant,Property Solutions India Pvt Ltd
8,'558924,Live,2024-09-24,NaT,2,Operations,Housekeeper,Property Solutions India Pvt Ltd
9,'541817,Live,2023-07-01,NaT,3,Operations,Housekeeper,Property Solutions India Pvt Ltd


In [61]:
print("Employee Status values:")
print(employee_master["Employee Status"].value_counts())

print("\nActual Join Date range:")
print("From:", employee_master["Actual Join Date"].min())
print("To  :", employee_master["Actual Join Date"].max())

print("\nResignation Date range:")
print("From:", employee_master["Resignation Date"].min())
print("To  :", employee_master["Resignation Date"].max())

Employee Status values:
Employee Status
Live         20339
On-Notice      147
Name: count, dtype: int64

Actual Join Date range:
From: 2004-06-01 00:00:00
To  : 2026-08-31 00:00:00

Resignation Date range:
From: 2025-12-07 00:00:00
To  : 2026-08-07 00:00:00


In [62]:
employee_master["Actual Join Date"] = pd.to_datetime(
    employee_master["Actual Join Date"],
    errors="coerce"
)

employee_master["Join Year"] = employee_master["Actual Join Date"].dt.year

yearly_joiners = (
    employee_master.groupby("Join Year")
    .size()
    .reset_index(name="New Joiners")
)

print(yearly_joiners.to_string(index=False))

 Join Year  New Joiners
      2004            1
      2005            2
      2006            3
      2007            5
      2008            8
      2009           21
      2010           42
      2011           39
      2012           27
      2013           57
      2014           66
      2015           82
      2016          275
      2017          192
      2018          379
      2019          378
      2020          367
      2021          688
      2022         1353
      2023         2082
      2024         2911
      2025         4284
      2026         7224


In [63]:
# Create monthly hiring history
monthly_hiring = (
    employee_master
    .set_index("Actual Join Date")
    .resample("MS")
    .size()
    .reset_index(name="New Joiners")
)

monthly_hiring.rename(columns={"Actual Join Date": "Month"}, inplace=True)

print("Monthly hiring history:")
print("From:", monthly_hiring["Month"].min())
print("To  :", monthly_hiring["Month"].max())
print("Number of months:", len(monthly_hiring))

display(monthly_hiring.tail(15))

Monthly hiring history:
From: 2004-06-01 00:00:00
To  : 2026-08-01 00:00:00
Number of months: 267


,Month,New Joiners
252,2025-06-01,486
253,2025-07-01,453
254,2025-08-01,455
255,2025-09-01,353
256,2025-10-01,296
257,2025-11-01,425
258,2025-12-01,485
259,2026-01-01,471
260,2026-02-01,583
261,2026-03-01,578


In [64]:

# Keep only complete months for model training
monthly_hiring = monthly_hiring[
    monthly_hiring["Month"] < pd.Timestamp("2026-08-01")
].copy()

print("Training data period:")
print("From:", monthly_hiring["Month"].min())
print("To  :", monthly_hiring["Month"].max())

print("\nNumber of complete months:", len(monthly_hiring))

display(monthly_hiring.tail(12))

Training data period:
From: 2004-06-01 00:00:00
To  : 2026-07-01 00:00:00

Number of complete months: 266


,Month,New Joiners
254,2025-08-01,455
255,2025-09-01,353
256,2025-10-01,296
257,2025-11-01,425
258,2025-12-01,485
259,2026-01-01,471
260,2026-02-01,583
261,2026-03-01,578
262,2026-04-01,1229
263,2026-05-01,1034


In [65]:
# Create monthly headcount history

months = pd.date_range(
    start=monthly_hiring["Month"].min(),
    end=monthly_hiring["Month"].max(),
    freq="MS"
)

monthly_headcount = []

for month in months:
    month_end = month + pd.offsets.MonthEnd(0)

    active_employees = (
        (employee_master["Actual Join Date"] <= month_end) &
        (
            employee_master["Resignation Date"].isna() |
            (employee_master["Resignation Date"] > month_end)
        )
    ).sum()

    monthly_headcount.append({
        "Month": month,
        "Headcount": active_employees
    })

monthly_headcount = pd.DataFrame(monthly_headcount)

print("Monthly headcount created.")
display(monthly_headcount.tail(12))

Monthly headcount created.


,Month,Headcount
254,2025-08-01,11703
255,2025-09-01,12056
256,2025-10-01,12352
257,2025-11-01,12777
258,2025-12-01,13261
259,2026-01-01,13731
260,2026-02-01,14311
261,2026-03-01,14882
262,2026-04-01,16104
263,2026-05-01,17028


In [66]:
# Combine monthly hiring and headcount data
workforce_data = pd.merge(
    monthly_hiring,
    monthly_headcount,
    on="Month",
    how="left"
)

# Sort by month
workforce_data = workforce_data.sort_values("Month").reset_index(drop=True)

print("Workforce dataset shape:", workforce_data.shape)

display(workforce_data.tail(12))

Workforce dataset shape: (266, 3)


,Month,New Joiners,Headcount
254,2025-08-01,455,11703
255,2025-09-01,353,12056
256,2025-10-01,296,12352
257,2025-11-01,425,12777
258,2025-12-01,485,13261
259,2026-01-01,471,13731
260,2026-02-01,583,14311
261,2026-03-01,578,14882
262,2026-04-01,1229,16104
263,2026-05-01,1034,17028


In [67]:
# Create monthly resignation history
resignation_data = (
    employee_master.dropna(subset=["Resignation Date"])
    .set_index("Resignation Date")
    .resample("MS")
    .size()
    .reset_index(name="Resigned Employees")
)

resignation_data.rename(columns={"Resignation Date": "Month"}, inplace=True)

print("Monthly resignation history:")
print("From:", resignation_data["Month"].min())
print("To  :", resignation_data["Month"].max())

display(resignation_data.tail(12))

Monthly resignation history:
From: 2025-12-01 00:00:00
To  : 2026-08-01 00:00:00


,Month,Resigned Employees
0,2025-12-01,1
1,2026-01-01,1
2,2026-02-01,3
3,2026-03-01,7
4,2026-04-01,7
5,2026-05-01,110
6,2026-06-01,19
7,2026-07-01,37
8,2026-08-01,69


In [68]:
# Merge resignation data with workforce data
workforce_data = pd.merge(
    workforce_data,
    resignation_data,
    on="Month",
    how="left"
)

# Keep missing resignation history as NaN for now
# Do NOT replace it with 0 because earlier months have no resignation data.
workforce_data["Turnover Rate (%)"] = (
    workforce_data["Resigned Employees"]
    / workforce_data["Headcount"]
) * 100

display(workforce_data.tail(15))

,Month,New Joiners,Headcount,Resigned Employees,Turnover Rate (%)
251,2025-05-01,380,10309,NaN,NaN
252,2025-06-01,486,10795,NaN,NaN
253,2025-07-01,453,11248,NaN,NaN
254,2025-08-01,455,11703,NaN,NaN
255,2025-09-01,353,12056,NaN,NaN
256,2025-10-01,296,12352,NaN,NaN
257,2025-11-01,425,12777,NaN,NaN
258,2025-12-01,485,13261,1.0,0.007541
259,2026-01-01,471,13731,1.0,0.007283
260,2026-02-01,583,14311,3.0,0.020963


In [69]:
# Calculate average employee tenure for each month

monthly_tenure = []

for month in workforce_data["Month"]:
    month_end = month + pd.offsets.MonthEnd(0)

    active = employee_master[
        (employee_master["Actual Join Date"] <= month_end) &
        (
            employee_master["Resignation Date"].isna() |
            (employee_master["Resignation Date"] > month_end)
        )
    ].copy()

    # Calculate tenure in years as of month-end
    tenure_years = (
        (month_end - active["Actual Join Date"]).dt.days / 365.25
    )

    monthly_tenure.append({
        "Month": month,
        "Average Tenure Years": tenure_years.mean()
    })

monthly_tenure = pd.DataFrame(monthly_tenure)

print("Average tenure calculated.")
display(monthly_tenure.tail(12))

Average tenure calculated.


,Month,Average Tenure Years
254,2025-08-01,2.666424
255,2025-09-01,2.669445
256,2025-10-01,2.689486
257,2025-11-01,2.680960
258,2025-12-01,2.666545
259,2026-01-01,2.658769
260,2026-02-01,2.626061
261,2026-03-01,2.608114
262,2026-04-01,2.489191
263,2026-05-01,2.428919


In [70]:
# Find all attendance files
attendance_files = [
    f for f in input_file_path.iterdir()
    if "Attendance" in f.name and f.suffix.lower() in [".csv", ".xlsx", ".xls"]
]

for file in attendance_files:
    print("\n" + "=" * 80)
    print("FILE:", file.name)
    
    if file.suffix.lower() == ".csv":
        temp = pd.read_csv(file, low_memory=False)
    else:
        temp = pd.read_excel(file)
    
    print("Shape:", temp.shape)
    print("Columns:")
    print(temp.columns.tolist())


FILE: Employee Attendance Report - PSIPL_20260826140835572.csv
Shape: (7820, 67)
Columns:
['Employee Code', 'Employee Name', 'Band', 'Grade', 'Division', 'Department', 'Designation', 'Date of joining', 'Date of Leaving', 'Employee Status', 'State', 'City', 'Branch', 'Employee Pay cycle', 'Residence', 'Cost Center', '26 JUL 26', '27 JUL 26', '28 JUL 26', '29 JUL 26', '30 JUL 26', '31 JUL 26', '01 AUG 26', '02 AUG 26', '03 AUG 26', '04 AUG 26', '05 AUG 26', '06 AUG 26', '07 AUG 26', '08 AUG 26', '09 AUG 26', '10 AUG 26', '11 AUG 26', '12 AUG 26', '13 AUG 26', '14 AUG 26', '15 AUG 26', '16 AUG 26', '17 AUG 26', '18 AUG 26', '19 AUG 26', '20 AUG 26', '21 AUG 26', '22 AUG 26', '23 AUG 26', '24 AUG 26', '25 AUG 26', 'Loss of Pay', 'Abscond', 'Site Holiday', 'Absent Count', 'Half Day Count', 'Present Count', 'Outdoor Duty', 'Worked On Off Count', 'Week Off Count', 'Public Holiday Count', 'Approved Leaves', 'BL Approved', 'BL Submit', 'EL Approved', 'EL Submit', 'PL Approved', 'PL Submit', 'P

In [71]:
# Load June and July attendance files

june_file = input_file_path / "Employee Attendance Report_20260826132240943.csv"
july_file = input_file_path / "Employee Attendance Report_20260826125133276.csv"

june_attendance = pd.read_csv(june_file, low_memory=False)
july_attendance = pd.read_csv(july_file, low_memory=False)

print("June shape:", june_attendance.shape)
print("July shape:", july_attendance.shape)

print("\nJune attendance totals:")
print(june_attendance[
    ["Present Count", "Absent Count", "Half Day Count",
     "Approved Leaves", "Payable", "Non Payable", "Total Days"]
].sum(numeric_only=True))

print("\nJuly attendance totals:")
print(july_attendance[
    ["Present Count", "Absent Count", "Half Day Count",
     "Approved Leaves", "Payable", "Non Payable", "Total Days"]
].sum(numeric_only=True))

June shape: (19569, 69)
July shape: (20069, 72)

June attendance totals:
Present Count      446493.0
Absent Count        35175.5
Half Day Count        419.0
Approved Leaves      1088.0
Payable            508218.5
Non Payable         47246.5
Total Days         555465.0
dtype: float64

July attendance totals:
Present Count      473993.5
Absent Count        43841.0
Half Day Count        498.0
Approved Leaves      1302.5
Payable            539370.0
Non Payable         58197.0
Total Days         597567.0
dtype: float64


In [72]:
# Calculate monthly attendance-based workforce factors

def calculate_attendance_metrics(df, month):
    total_days = df["Total Days"].sum()
    absent_days = df["Absent Count"].sum()
    half_days = df["Half Day Count"].sum()

    # Count a half-day as 0.5 absent day
    absence_days = absent_days + (0.5 * half_days)

    absenteeism_rate = (absence_days / total_days) * 100

    return {
        "Month": pd.Timestamp(month),
        "Absenteeism Rate (%)": absenteeism_rate,
        "Unplanned Leave Days": absent_days
    }


june_metrics = calculate_attendance_metrics(
    june_attendance, "2026-06-01"
)

july_metrics = calculate_attendance_metrics(
    july_attendance, "2026-07-01"
)

attendance_metrics = pd.DataFrame([
    june_metrics,
    july_metrics
])

print("Attendance metrics:")
display(attendance_metrics)

Attendance metrics:


,Month,Absenteeism Rate (%),Unplanned Leave Days
0,2026-06-01,6.370338,35175.5
1,2026-07-01,7.378252,43841.0


In [73]:
# Find all overtime files
overtime_files = [
    f for f in input_file_path.iterdir()
    if "Overtime" in f.name and f.suffix.lower() in [".csv", ".xlsx", ".xls"]
]

for file in overtime_files:
    print("\n" + "=" * 80)
    print("FILE:", file.name)

    temp = pd.read_csv(file, low_memory=False) if file.suffix.lower() == ".csv" else pd.read_excel(file)

    print("Shape:", temp.shape)
    print("Columns:")
    print(temp.columns.tolist())


FILE: Overtime Upload Report_20260807084943902.csv
Shape: (3228, 109)
Columns:
['EMP CODE', 'EMP NAME', 'BRANCH NAME', 'CITY NAME', 'JOIN DATE', 'LAST WORKING DATE', 'DESIGNATION', 'EMPLOYEE CATEGORY', 'COST CENTRE CODE', 'COST CENTRE NAME', 'SITE CODE', 'SITE NAME', 'PAY CYCLE', '26-JUN-26_CH', '26-JUN-26_NCH', '26-JUN-26_TOT', '27-JUN-26_CH', '27-JUN-26_NCH', '27-JUN-26_TOT', '28-JUN-26_CH', '28-JUN-26_NCH', '28-JUN-26_TOT', '29-JUN-26_CH', '29-JUN-26_NCH', '29-JUN-26_TOT', '30-JUN-26_CH', '30-JUN-26_NCH', '30-JUN-26_TOT', '01-JUL-26_CH', '01-JUL-26_NCH', '01-JUL-26_TOT', '02-JUL-26_CH', '02-JUL-26_NCH', '02-JUL-26_TOT', '03-JUL-26_CH', '03-JUL-26_NCH', '03-JUL-26_TOT', '04-JUL-26_CH', '04-JUL-26_NCH', '04-JUL-26_TOT', '05-JUL-26_CH', '05-JUL-26_NCH', '05-JUL-26_TOT', '06-JUL-26_CH', '06-JUL-26_NCH', '06-JUL-26_TOT', '07-JUL-26_CH', '07-JUL-26_NCH', '07-JUL-26_TOT', '08-JUL-26_CH', '08-JUL-26_NCH', '08-JUL-26_TOT', '09-JUL-26_CH', '09-JUL-26_NCH', '09-JUL-26_TOT', '10-JUL-26_CH', '1

In [74]:
# Load complete July overtime data

july_ot_file = input_file_path / "Overtime Upload Report_20260807085324157.csv"

july_ot = pd.read_csv(july_ot_file, low_memory=False)

# Make sure TOTAL is numeric
july_ot["TOTAL"] = pd.to_numeric(july_ot["TOTAL"], errors="coerce")

print("July OT shape:", july_ot.shape)

print("\nTOTAL OT summary:")
print(july_ot["TOTAL"].describe())

print("\nMissing TOTAL values:", july_ot["TOTAL"].isna().sum())

display(
    july_ot[["EMP CODE", "EMP NAME", "TOTAL"]].head(10)
)

July OT shape: (4783, 112)

TOTAL OT summary:
count    4783.000000
mean       11.439452
std        22.874620
min         0.000000
25%         0.000000
50%         0.000000
75%        16.000000
max       352.000000
Name: TOTAL, dtype: float64

Missing TOTAL values: 0


,EMP CODE,EMP NAME,TOTAL
0,26,Vinesh Gunaji Patade,17.0
1,1009,Sanjay Sitaram Jagtap,4.0
2,2044,Prashant Sirmanta Waghmare,0.0
3,5213,Gautam Chintaman Shirke,21.0
4,5568,Tabitha Rajkumar Surya,15.0
5,13622,Arvind Vasant Kadam,12.0
6,13971,Parshuram Shankar Mali,0.0
7,15126,Devendra Nandivadekar,37.0
8,16867,Gayatri Prajapati,10.0
9,19544,Shridhar Sakharam Kadam,16.0


In [75]:
# Validate July overtime TOTAL

print("TOTAL OT data type:", july_ot["TOTAL"].dtype)

print("\nNumber of employees with OT > 0:",
      (july_ot["TOTAL"] > 0).sum())

print("Number of employees with OT = 0:",
      (july_ot["TOTAL"] == 0).sum())

print("\nChargeable OT total:",
      pd.to_numeric(july_ot["CHARGEABLE TOTAL"], errors="coerce").sum())

print("Non-chargeable OT total:",
      pd.to_numeric(july_ot["NON CHARGEABLE TOTAL"], errors="coerce").sum())

print("TOTAL OT sum:",
      july_ot["TOTAL"].sum())

print("\nTop 10 employees by OT:")
display(
    july_ot[["EMP CODE", "EMP NAME", "CHARGEABLE TOTAL",
             "NON CHARGEABLE TOTAL", "TOTAL"]]
    .sort_values("TOTAL", ascending=False)
    .head(10)
)

TOTAL OT data type: float64

Number of employees with OT > 0: 2057
Number of employees with OT = 0: 2726

Chargeable OT total: 20506.8
Non-chargeable OT total: 34208.1
TOTAL OT sum: 54714.9

Top 10 employees by OT:


,EMP CODE,EMP NAME,CHARGEABLE TOTAL,NON CHARGEABLE TOTAL,TOTAL
374,521360,Sagar Navnath Jogi,0.0,352.0,352.0
1447,553682,Nilesh Gunaji Valam,0.0,224.0,224.0
3147,577910,Amol Chandrakant Kudekar,0.0,208.0,208.0
1042,543354,Vaniya Jaydip,0.0,208.0,208.0
1662,558364,Narayan Sakharam Raghav,0.0,200.0,200.0
4321,584430,SATISH KUMAR,184.0,0.0,184.0
3953,583021,KANIKA JAMADAR,0.0,184.0,184.0
174,86115,Suyash Shivram Ghadigaonkar,0.0,183.0,183.0
2939,576414,SARITA DEVI,0.0,176.0,176.0
2815,575198,MAHENDRA,0.0,176.0,176.0


In [76]:
# Create employee-level July overtime feature

july_ot_feature = july_ot[
    ["EMP CODE", "CHARGEABLE TOTAL", "NON CHARGEABLE TOTAL", "TOTAL"]
].copy()

july_ot_feature = july_ot_feature.rename(columns={
    "EMP CODE": "Employee Code",
    "CHARGEABLE TOTAL": "Chargeable OT",
    "NON CHARGEABLE TOTAL": "Non Chargeable OT",
    "TOTAL": "Total OT"
})

# Remove duplicate employee codes, if any
july_ot_feature = (
    july_ot_feature
    .groupby("Employee Code", as_index=False)[
        ["Chargeable OT", "Non Chargeable OT", "Total OT"]
    ]
    .sum()
)

print("July OT feature shape:", july_ot_feature.shape)

display(july_ot_feature.head(10))

July OT feature shape: (4781, 4)


,Employee Code,Chargeable OT,Non Chargeable OT,Total OT
0,1009,4.0,0.0,4.0
1,13622,12.0,0.0,12.0
2,13971,0.0,0.0,0.0
3,15126,37.0,0.0,37.0
4,16867,0.0,10.0,10.0
5,19544,0.0,16.0,16.0
6,2044,0.0,0.0,0.0
7,21332,0.0,0.0,0.0
8,21978,0.0,9.0,9.0
9,22095,19.0,0.0,19.0


In [77]:
# Load June overtime data

june_ot_path = next(
    input_file_path.glob("*Overtime Upload Report_20260807084943902.csv")
)

june_ot = pd.read_csv(june_ot_path)

print("June OT shape:", june_ot.shape)
print("\nJune OT columns:")
print(june_ot.columns.tolist())

print("\nJune TOTAL OT summary:")
print(june_ot["TOTAL"].describe())

print("\nMissing TOTAL values:", june_ot["TOTAL"].isna().sum())

June OT shape: (3228, 109)

June OT columns:
['EMP CODE', 'EMP NAME', 'BRANCH NAME', 'CITY NAME', 'JOIN DATE', 'LAST WORKING DATE', 'DESIGNATION', 'EMPLOYEE CATEGORY', 'COST CENTRE CODE', 'COST CENTRE NAME', 'SITE CODE', 'SITE NAME', 'PAY CYCLE', '26-JUN-26_CH', '26-JUN-26_NCH', '26-JUN-26_TOT', '27-JUN-26_CH', '27-JUN-26_NCH', '27-JUN-26_TOT', '28-JUN-26_CH', '28-JUN-26_NCH', '28-JUN-26_TOT', '29-JUN-26_CH', '29-JUN-26_NCH', '29-JUN-26_TOT', '30-JUN-26_CH', '30-JUN-26_NCH', '30-JUN-26_TOT', '01-JUL-26_CH', '01-JUL-26_NCH', '01-JUL-26_TOT', '02-JUL-26_CH', '02-JUL-26_NCH', '02-JUL-26_TOT', '03-JUL-26_CH', '03-JUL-26_NCH', '03-JUL-26_TOT', '04-JUL-26_CH', '04-JUL-26_NCH', '04-JUL-26_TOT', '05-JUL-26_CH', '05-JUL-26_NCH', '05-JUL-26_TOT', '06-JUL-26_CH', '06-JUL-26_NCH', '06-JUL-26_TOT', '07-JUL-26_CH', '07-JUL-26_NCH', '07-JUL-26_TOT', '08-JUL-26_CH', '08-JUL-26_NCH', '08-JUL-26_TOT', '09-JUL-26_CH', '09-JUL-26_NCH', '09-JUL-26_TOT', '10-JUL-26_CH', '10-JUL-26_NCH', '10-JUL-26_TOT', '11

In [78]:
# Create employee-level June overtime feature

june_ot_feature = june_ot[
    ["EMP CODE", "CHARGEABLE TOTAL", "NON CHARGEABLE TOTAL", "TOTAL"]
].copy()

june_ot_feature = june_ot_feature.rename(columns={
    "EMP CODE": "Employee Code",
    "CHARGEABLE TOTAL": "Chargeable OT June",
    "NON CHARGEABLE TOTAL": "Non Chargeable OT June",
    "TOTAL": "Total OT June"
})

# One row per employee
june_ot_feature = (
    june_ot_feature
    .groupby("Employee Code", as_index=False)[
        ["Chargeable OT June",
         "Non Chargeable OT June",
         "Total OT June"]
    ]
    .sum()
)

print("June OT feature shape:", june_ot_feature.shape)

display(june_ot_feature.head(10))

June OT feature shape: (3218, 4)


,Employee Code,Chargeable OT June,Non Chargeable OT June,Total OT June
0,11168,0.0,104.0,104.0
1,11603,0.0,0.0,0.0
2,11625,104.0,0.0,104.0
3,14000,0.0,56.0,56.0
4,16145,6.0,0.0,6.0
5,16166,21.0,0.0,21.0
6,17288,4.0,0.0,4.0
7,17366,13.0,0.0,13.0
8,17485,0.0,176.0,176.0
9,18658,40.0,40.0,80.0


In [79]:
# Fix Employee Code datatype before merging

june_ot_feature["Employee Code"] = (
    june_ot_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

july_ot_feature["Employee Code"] = (
    july_ot_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# Combine June and July OT
ot_features = pd.merge(
    june_ot_feature,
    july_ot_feature,
    on="Employee Code",
    how="outer"
)

# Employees without OT in one month = 0
ot_cols = [
    "Chargeable OT June",
    "Non Chargeable OT June",
    "Total OT June",
    "Chargeable OT",
    "Non Chargeable OT",
    "Total OT"
]

ot_features[ot_cols] = ot_features[ot_cols].fillna(0)

# Change in overtime from June to July
ot_features["OT Change"] = (
    ot_features["Total OT"] - ot_features["Total OT June"]
)

print("Combined OT feature shape:", ot_features.shape)

display(ot_features.head(10))

Combined OT feature shape: (7999, 8)


,Employee Code,Chargeable OT June,Non Chargeable OT June,Total OT June,Chargeable OT,Non Chargeable OT,Total OT,OT Change
0,1009,0.0,0.0,0.0,4.0,0.0,4.0,4.0
1,11168,0.0,104.0,104.0,0.0,0.0,0.0,-104.0
2,11603,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,11625,104.0,0.0,104.0,0.0,0.0,0.0,-104.0
4,13622,0.0,0.0,0.0,12.0,0.0,12.0,12.0
5,13971,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,14000,0.0,56.0,56.0,0.0,0.0,0.0,-56.0
7,15126,0.0,0.0,0.0,37.0,0.0,37.0,37.0
8,16145,6.0,0.0,6.0,0.0,0.0,0.0,-6.0
9,16166,21.0,0.0,21.0,0.0,0.0,0.0,-21.0


In [80]:
# Create employee-level June attendance features

june_att_feature = june_attendance[
    [
        "Employee Code",
        "Present Count",
        "Absent Count",
        "Half Day Count",
        "Approved Leaves",
        "Payable",
        "Non Payable",
        "Total Days"
    ]
].copy()

# Standardize Employee Code
june_att_feature["Employee Code"] = (
    june_att_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# Convert attendance fields to numeric
attendance_cols = [
    "Present Count",
    "Absent Count",
    "Half Day Count",
    "Approved Leaves",
    "Payable",
    "Non Payable",
    "Total Days"
]

for col in attendance_cols:
    june_att_feature[col] = pd.to_numeric(
        june_att_feature[col], errors="coerce"
    ).fillna(0)

# Aggregate to one row per employee
june_att_feature = (
    june_att_feature
    .groupby("Employee Code", as_index=False)[attendance_cols]
    .sum()
)

# Calculate June absence rate
june_att_feature["Absence Rate June"] = (
    june_att_feature["Absent Count"] /
    june_att_feature["Total Days"].replace(0, np.nan)
    * 100
).fillna(0)

print("June attendance feature shape:", june_att_feature.shape)

display(june_att_feature.head(10))

June attendance feature shape: (19569, 9)


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate June
0,00026,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
1,00349,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
2,01009,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
3,02044,26.0,0.0,0,0.0,30.0,0.0,30,0.000000
4,05213,26.0,0.0,0,0.0,30.0,0.0,30,0.000000
5,05568,22.5,3.5,1,0.0,26.5,3.5,30,11.666667
6,06286,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
7,10145,27.0,0.0,0,0.0,30.0,0.0,30,0.000000
8,11006,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
9,11168,26.0,0.0,0,0.0,30.0,0.0,30,0.000000


In [81]:
# Create employee-level June attendance features

june_att_feature = june_attendance[
    [
        "Employee Code",
        "Present Count",
        "Absent Count",
        "Half Day Count",
        "Approved Leaves",
        "Payable",
        "Non Payable",
        "Total Days"
    ]
].copy()

# Standardize Employee Code
june_att_feature["Employee Code"] = (
    june_att_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)
)

# Convert attendance fields to numeric
attendance_cols = [
    "Present Count",
    "Absent Count",
    "Half Day Count",
    "Approved Leaves",
    "Payable",
    "Non Payable",
    "Total Days"
]

for col in attendance_cols:
    june_att_feature[col] = pd.to_numeric(
        june_att_feature[col], errors="coerce"
    ).fillna(0)

# Aggregate to one row per employee
june_att_feature = (
    june_att_feature
    .groupby("Employee Code", as_index=False)[attendance_cols]
    .sum()
)

# Calculate June absence rate
june_att_feature["Absence Rate June"] = (
    june_att_feature["Absent Count"] /
    june_att_feature["Total Days"].replace(0, np.nan)
    * 100
).fillna(0)

print("June attendance feature shape:", june_att_feature.shape)

display(june_att_feature.head(10))

June attendance feature shape: (19569, 9)


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate June
0,00026,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
1,00349,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
2,01009,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
3,02044,26.0,0.0,0,0.0,30.0,0.0,30,0.000000
4,05213,26.0,0.0,0,0.0,30.0,0.0,30,0.000000
5,05568,22.5,3.5,1,0.0,26.5,3.5,30,11.666667
6,06286,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
7,10145,27.0,0.0,0,0.0,30.0,0.0,30,0.000000
8,11006,30.0,0.0,0,0.0,30.0,0.0,30,0.000000
9,11168,26.0,0.0,0,0.0,30.0,0.0,30,0.000000


In [82]:
# Normalize Employee Code consistently across attendance and OT features

def normalize_employee_code(df, col="Employee Code"):
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.lstrip("0")
    )
    
    # Avoid empty string for codes that were all zeros
    df[col] = df[col].replace("", "0")
    
    return df


june_att_feature = normalize_employee_code(june_att_feature)
ot_features = normalize_employee_code(ot_features)

print("June attendance codes:", june_att_feature["Employee Code"].nunique())
print("OT feature codes:", ot_features["Employee Code"].nunique())

display(june_att_feature.head())

June attendance codes: 19569
OT feature codes: 7999


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate June
0,26,30.0,0.0,0,0.0,30.0,0.0,30,0.0
1,349,30.0,0.0,0,0.0,30.0,0.0,30,0.0
2,1009,30.0,0.0,0,0.0,30.0,0.0,30,0.0
3,2044,26.0,0.0,0,0.0,30.0,0.0,30,0.0
4,5213,26.0,0.0,0,0.0,30.0,0.0,30,0.0


In [83]:
# ============================================================
# Create employee-level July attendance features
# ============================================================

# Make a copy so the original July attendance data is not changed
july_att_feature = july_attendance.copy()

# Normalize Employee Code
july_att_feature["Employee Code"] = (
    july_att_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.lstrip("0")
)

# Avoid blank employee codes
july_att_feature["Employee Code"] = (
    july_att_feature["Employee Code"].replace("", "0")
)

# Columns required for the model
july_att_feature = july_att_feature[
    [
        "Employee Code",
        "Present Count",
        "Absent Count",
        "Half Day Count",
        "Approved Leaves",
        "Payable",
        "Non Payable",
        "Total Days"
    ]
].copy()

# Convert attendance metrics to numeric
numeric_cols = [
    "Present Count",
    "Absent Count",
    "Half Day Count",
    "Approved Leaves",
    "Payable",
    "Non Payable",
    "Total Days"
]

for col in numeric_cols:
    july_att_feature[col] = pd.to_numeric(
        july_att_feature[col],
        errors="coerce"
    ).fillna(0)

# Calculate July absence rate
july_att_feature["Absence Rate July"] = (
    july_att_feature["Absent Count"]
    / july_att_feature["Total Days"].replace(0, np.nan)
) * 100

july_att_feature["Absence Rate July"] = (
    july_att_feature["Absence Rate July"]
    .fillna(0)
)

# Remove duplicate employees, if any
july_att_feature = (
    july_att_feature
    .drop_duplicates(subset=["Employee Code"])
    .reset_index(drop=True)
)

# Check output
print("July attendance feature shape:", july_att_feature.shape)
print("July attendance employee codes:",
      july_att_feature["Employee Code"].nunique())

display(july_att_feature.head(10))

July attendance feature shape: (20069, 9)
July attendance employee codes: 20069


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate July
0,'577363,23.0,0.0,0,0.0,31.0,0.0,31,0.000000
1,'577403,3.0,1.0,0,0.0,3.0,1.0,4,25.000000
2,'577375,0.0,4.0,0,0.0,0.0,31.0,31,12.903226
3,'577391,22.0,9.0,0,0.0,22.0,9.0,31,29.032258
4,'577335,27.0,0.0,0,0.0,31.0,0.0,31,0.000000
5,'576990,4.0,20.0,0,0.0,4.0,20.0,24,83.333333
6,'576994,0.0,4.0,0,0.0,0.0,31.0,31,12.903226
7,'576993,28.0,2.0,0,0.0,29.0,2.0,31,6.451613
8,'576946,26.0,1.0,0,0.0,30.0,1.0,31,3.225806
9,'576945,27.0,0.0,0,0.0,31.0,0.0,31,0.000000


In [84]:
# ============================================================
# Fix July Employee Code format
# ============================================================

july_att_feature["Employee Code"] = (
    july_att_feature["Employee Code"]
    .astype(str)
    .str.strip()
    .str.replace("'", "", regex=False)   # remove leading apostrophe
    .str.replace(".0", "", regex=False)
    .str.lstrip("0")                     # remove leading zeros
)

# Avoid blank employee codes
july_att_feature["Employee Code"] = (
    july_att_feature["Employee Code"].replace("", "0")
)

print("July attendance feature shape:", july_att_feature.shape)
print(
    "July attendance employee codes:",
    july_att_feature["Employee Code"].nunique()
)

display(july_att_feature.head(10))

July attendance feature shape: (20069, 9)


July attendance employee codes: 20069


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate July
0,577363,23.0,0.0,0,0.0,31.0,0.0,31,0.000000
1,577403,3.0,1.0,0,0.0,3.0,1.0,4,25.000000
2,577375,0.0,4.0,0,0.0,0.0,31.0,31,12.903226
3,577391,22.0,9.0,0,0.0,22.0,9.0,31,29.032258
4,577335,27.0,0.0,0,0.0,31.0,0.0,31,0.000000
5,576990,4.0,20.0,0,0.0,4.0,20.0,24,83.333333
6,576994,0.0,4.0,0,0.0,0.0,31.0,31,12.903226
7,576993,28.0,2.0,0,0.0,29.0,2.0,31,6.451613
8,576946,26.0,1.0,0,0.0,30.0,1.0,31,3.225806
9,576945,27.0,0.0,0,0.0,31.0,0.0,31,0.000000


In [85]:
# ============================================================
# Validate Employee Codes across all feature datasets
# ============================================================

datasets = {
    "June Attendance": june_att_feature,
    "July Attendance": july_att_feature,
    "OT Features": ot_features
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Unique Employee Codes:", df["Employee Code"].nunique())
    print("Duplicate Employee Codes:", df["Employee Code"].duplicated().sum())
    print("Employee Code dtype:", df["Employee Code"].dtype)

# ------------------------------------------------------------
# Check overlap between attendance and OT
# ------------------------------------------------------------

june_codes = set(june_att_feature["Employee Code"])
july_codes = set(july_att_feature["Employee Code"])
ot_codes = set(ot_features["Employee Code"])

print("\n" + "=" * 60)
print("EMPLOYEE CODE OVERLAP CHECK")
print("=" * 60)

print("June Attendance ∩ July Attendance:",
      len(june_codes & july_codes))

print("June Attendance ∩ OT:",
      len(june_codes & ot_codes))

print("July Attendance ∩ OT:",
      len(july_codes & ot_codes))

print("Present in ALL THREE:",
      len(june_codes & july_codes & ot_codes))


June Attendance
Rows: 19569
Unique Employee Codes: 19569
Duplicate Employee Codes: 0
Employee Code dtype: str

July Attendance
Rows: 20069
Unique Employee Codes: 20069
Duplicate Employee Codes: 0
Employee Code dtype: str

OT Features
Rows: 7999
Unique Employee Codes: 7999
Duplicate Employee Codes: 0
Employee Code dtype: str

EMPLOYEE CODE OVERLAP CHECK
June Attendance ∩ July Attendance: 18438
June Attendance ∩ OT: 7551
July Attendance ∩ OT: 7970
Present in ALL THREE: 7523


In [86]:
# ============================================================
# Combine June + July Attendance + June/July OT features
# July attendance is the base/current workforce population
# ============================================================

# Make copies
final_features = july_att_feature.copy()

# ------------------------------------------------------------
# Rename June attendance columns
# ------------------------------------------------------------

june_att_merge = june_att_feature.rename(columns={
    "Present Count": "Present Count June",
    "Absent Count": "Absent Count June",
    "Half Day Count": "Half Day Count June",
    "Approved Leaves": "Approved Leaves June",
    "Payable": "Payable June",
    "Non Payable": "Non Payable June",
    "Total Days": "Total Days June"
})

# ------------------------------------------------------------
# Merge June attendance onto July employees
# ------------------------------------------------------------

final_features = final_features.merge(
    june_att_merge,
    on="Employee Code",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Merge OT features
# ------------------------------------------------------------

final_features = final_features.merge(
    ot_features,
    on="Employee Code",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Fill missing OT values with 0
# ------------------------------------------------------------

ot_columns = [
    "Chargeable OT June",
    "Non Chargeable OT June",
    "Total OT June",
    "Chargeable OT",
    "Non Chargeable OT",
    "Total OT",
    "OT Change"
]

for col in ot_columns:
    if col in final_features.columns:
        final_features[col] = (
            pd.to_numeric(final_features[col], errors="coerce")
            .fillna(0)
        )

# ------------------------------------------------------------
# Check June attendance missingness
# ------------------------------------------------------------

june_columns = [
    "Present Count June",
    "Absent Count June",
    "Half Day Count June",
    "Approved Leaves June",
    "Payable June",
    "Non Payable June",
    "Total Days June"
]

print("Final feature shape:", final_features.shape)

print(
    "July employees without June attendance:",
    final_features["Present Count June"].isna().sum()
)

print(
    "July employees without OT record:",
    final_features["Total OT"].eq(0).sum()
)

print("\nFinal feature columns:")
print(final_features.columns.tolist())

display(final_features.head(10))

Final feature shape: (20069, 24)
July employees without June attendance: 1631
July employees without OT record: 18014

Final feature columns:
['Employee Code', 'Present Count', 'Absent Count', 'Half Day Count', 'Approved Leaves', 'Payable', 'Non Payable', 'Total Days', 'Absence Rate July', 'Present Count June', 'Absent Count June', 'Half Day Count June', 'Approved Leaves June', 'Payable June', 'Non Payable June', 'Total Days June', 'Absence Rate June', 'Chargeable OT June', 'Non Chargeable OT June', 'Total OT June', 'Chargeable OT', 'Non Chargeable OT', 'Total OT', 'OT Change']


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate July,Present Count June,...,Non Payable June,Total Days June,Absence Rate June,Chargeable OT June,Non Chargeable OT June,Total OT June,Chargeable OT,Non Chargeable OT,Total OT,OT Change
0,577363,23.0,0.0,0,0.0,31.0,0.0,31,0.000000,26.0,...,0.0,30.0,0.000000,0.0,0.0,0.0,0.0,56.0,56.0,56.0
1,577403,3.0,1.0,0,0.0,3.0,1.0,4,25.000000,27.0,...,3.0,30.0,10.000000,24.0,0.0,24.0,0.0,0.0,0.0,-24.0
2,577375,0.0,4.0,0,0.0,0.0,31.0,31,12.903226,0.0,...,30.0,30.0,13.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,577391,22.0,9.0,0,0.0,22.0,9.0,31,29.032258,13.0,...,17.0,30.0,56.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,577335,27.0,0.0,0,0.0,31.0,0.0,31,0.000000,22.0,...,4.0,30.0,13.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,576990,4.0,20.0,0,0.0,4.0,20.0,24,83.333333,27.0,...,0.0,30.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,576994,0.0,4.0,0,0.0,0.0,31.0,31,12.903226,0.0,...,30.0,30.0,13.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,576993,28.0,2.0,0,0.0,29.0,2.0,31,6.451613,30.0,...,0.0,30.0,0.000000,4.0,0.0,4.0,0.0,0.0,0.0,-4.0
8,576946,26.0,1.0,0,0.0,30.0,1.0,31,3.225806,26.0,...,0.0,30.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,576945,27.0,0.0,0,0.0,31.0,0.0,31,0.000000,26.0,...,0.0,30.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
# ============================================================
# NORMALIZE EMPLOYEE CODE ACROSS ALL FEATURE DATASETS
# ============================================================

def normalize_employee_code(df, col="Employee Code"):
    df = df.copy()

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lstrip("0")
    )

    # Avoid empty string after removing leading zeros
    df[col] = df[col].replace("", "0")

    return df


june_att_feature = normalize_employee_code(june_att_feature)
july_att_feature = normalize_employee_code(july_att_feature)
ot_features = normalize_employee_code(ot_features)

print("June attendance codes:", june_att_feature["Employee Code"].nunique())
print("July attendance codes:", july_att_feature["Employee Code"].nunique())
print("OT feature codes:", ot_features["Employee Code"].nunique())

print("June duplicate codes:",
      june_att_feature["Employee Code"].duplicated().sum())

print("July duplicate codes:",
      july_att_feature["Employee Code"].duplicated().sum())

print("OT duplicate codes:",
      ot_features["Employee Code"].duplicated().sum())

June attendance codes: 19569
July attendance codes: 20069
OT feature codes: 7999
June duplicate codes: 0
July duplicate codes: 0
OT duplicate codes: 0


In [88]:
# ============================================================
# CREATE DATA AVAILABILITY INDICATORS
# ============================================================

june_codes = set(june_att_feature["Employee Code"])
ot_codes = set(ot_features["Employee Code"])

july_att_feature["June_Attendance_Available"] = (
    july_att_feature["Employee Code"].isin(june_codes).astype(int)
)

july_att_feature["OT_July_Available"] = (
    july_att_feature["Employee Code"].isin(ot_codes).astype(int)
)

print(
    "July employees with June attendance:",
    july_att_feature["June_Attendance_Available"].sum()
)

print(
    "July employees without June attendance:",
    (
        july_att_feature["June_Attendance_Available"].eq(0)
    ).sum()
)

print(
    "July employees with OT record:",
    july_att_feature["OT_July_Available"].sum()
)

print(
    "July employees without OT record:",
    (
        july_att_feature["OT_July_Available"].eq(0)
    ).sum()
)

July employees with June attendance: 18438
July employees without June attendance: 1631
July employees with OT record: 7970
July employees without OT record: 12099


In [89]:
# ============================================================
# MERGE JUNE ATTENDANCE INTO JULY EMPLOYEE BASE
# ============================================================

june_cols = [
    "Employee Code",
    "Present Count",
    "Absent Count",
    "Half Day Count",
    "Approved Leaves",
    "Payable",
    "Non Payable",
    "Total Days",
    "Absence Rate June"
]

june_merge = june_att_feature[june_cols].copy()

june_merge = june_merge.rename(columns={
    "Present Count": "Present Count June",
    "Absent Count": "Absent Count June",
    "Half Day Count": "Half Day Count June",
    "Approved Leaves": "Approved Leaves June",
    "Payable": "Payable June",
    "Non Payable": "Non Payable June",
    "Total Days": "Total Days June"
})

final_features = july_att_feature.merge(
    june_merge,
    on="Employee Code",
    how="left",
    validate="one_to_one"
)

print("After June merge:", final_features.shape)

After June merge: (20069, 19)


In [90]:
# ============================================================
# MERGE JUNE + JULY OT FEATURES
# ============================================================

ot_cols = [
    "Employee Code",
    "Chargeable OT June",
    "Non Chargeable OT June",
    "Total OT June",
    "Chargeable OT",
    "Non Chargeable OT",
    "Total OT"
]

ot_merge = ot_features[ot_cols].copy()

final_features = final_features.merge(
    ot_merge,
    on="Employee Code",
    how="left",
    validate="one_to_one"
)

print("After OT merge:", final_features.shape)

After OT merge: (20069, 25)


In [91]:
# ============================================================
# CALCULATE OT CHANGE
# ============================================================

# Employees with no June OT are treated as 0 June OT
final_features["Total OT June"] = (
    final_features["Total OT June"].fillna(0)
)

# Employees with no July OT record are treated as 0 July OT
final_features["Total OT"] = (
    final_features["Total OT"].fillna(0)
)

final_features["OT Change"] = (
    final_features["Total OT"] -
    final_features["Total OT June"]
)


In [92]:
ot_zero_cols = [
    "Chargeable OT June",
    "Non Chargeable OT June",
    "Chargeable OT",
    "Non Chargeable OT"
]

for col in ot_zero_cols:
    final_features[col] = final_features[col].fillna(0)

In [93]:
# ============================================================
# HANDLE MISSING JUNE ATTENDANCE
# ============================================================

june_att_cols = [
    "Present Count June",
    "Absent Count June",
    "Half Day Count June",
    "Approved Leaves June",
    "Payable June",
    "Non Payable June",
    "Total Days June",
    "Absence Rate June"
]

# Keep missing June values as NaN for employees
# who genuinely had no June attendance record.

print(
    "Employees without June attendance:",
    final_features["Present Count June"].isna().sum()
)


Employees without June attendance: 1631


In [94]:
# ============================================================
# FINAL FEATURE DATASET
# ============================================================

final_feature_cols = [
    "Employee Code",

    # July Attendance
    "Present Count",
    "Absent Count",
    "Half Day Count",
    "Approved Leaves",
    "Payable",
    "Non Payable",
    "Total Days",
    "Absence Rate July",

    # June Attendance
    "Present Count June",
    "Absent Count June",
    "Half Day Count June",
    "Approved Leaves June",
    "Payable June",
    "Non Payable June",
    "Total Days June",
    "Absence Rate June",
    "June_Attendance_Available",

    # June OT
    "Chargeable OT June",
    "Non Chargeable OT June",
    "Total OT June",

    # July OT
    "Chargeable OT",
    "Non Chargeable OT",
    "Total OT",
    "OT_July_Available",

    # OT movement
    "OT Change"
]

final_features = final_features[final_feature_cols].copy()

print("Final feature shape:", final_features.shape)
print("\nFinal feature columns:")
print(final_features.columns.tolist())

display(final_features.head(10))

Final feature shape: (20069, 26)

Final feature columns:
['Employee Code', 'Present Count', 'Absent Count', 'Half Day Count', 'Approved Leaves', 'Payable', 'Non Payable', 'Total Days', 'Absence Rate July', 'Present Count June', 'Absent Count June', 'Half Day Count June', 'Approved Leaves June', 'Payable June', 'Non Payable June', 'Total Days June', 'Absence Rate June', 'June_Attendance_Available', 'Chargeable OT June', 'Non Chargeable OT June', 'Total OT June', 'Chargeable OT', 'Non Chargeable OT', 'Total OT', 'OT_July_Available', 'OT Change']


,Employee Code,Present Count,Absent Count,Half Day Count,Approved Leaves,Payable,Non Payable,Total Days,Absence Rate July,Present Count June,...,Absence Rate June,June_Attendance_Available,Chargeable OT June,Non Chargeable OT June,Total OT June,Chargeable OT,Non Chargeable OT,Total OT,OT_July_Available,OT Change
0,577363,23.0,0.0,0,0.0,31.0,0.0,31,0.000000,26.0,...,0.000000,1,0.0,0.0,0.0,0.0,56.0,56.0,1,56.0
1,577403,3.0,1.0,0,0.0,3.0,1.0,4,25.000000,27.0,...,10.000000,1,24.0,0.0,24.0,0.0,0.0,0.0,1,-24.0
2,577375,0.0,4.0,0,0.0,0.0,31.0,31,12.903226,0.0,...,13.333333,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
3,577391,22.0,9.0,0,0.0,22.0,9.0,31,29.032258,13.0,...,56.666667,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0
4,577335,27.0,0.0,0,0.0,31.0,0.0,31,0.000000,22.0,...,13.333333,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
5,576990,4.0,20.0,0,0.0,4.0,20.0,24,83.333333,27.0,...,0.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0
6,576994,0.0,4.0,0,0.0,0.0,31.0,31,12.903226,0.0,...,13.333333,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
7,576993,28.0,2.0,0,0.0,29.0,2.0,31,6.451613,30.0,...,0.000000,1,4.0,0.0,4.0,0.0,0.0,0.0,1,-4.0
8,576946,26.0,1.0,0,0.0,30.0,1.0,31,3.225806,26.0,...,0.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
9,576945,27.0,0.0,0,0.0,31.0,0.0,31,0.000000,26.0,...,0.000000,1,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0


In [95]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 70)
print("FINAL FEATURE VALIDATION")
print("=" * 70)

print("Rows:", len(final_features))
print("Columns:", len(final_features.columns))

print("\nUnique employees:",
      final_features["Employee Code"].nunique())

print("Duplicate employee codes:",
      final_features["Employee Code"].duplicated().sum())

print("\nJune attendance available:",
      final_features["June_Attendance_Available"].sum())

print("June attendance unavailable:",
      (final_features["June_Attendance_Available"] == 0).sum())

print("\nJuly OT available:",
      final_features["OT_July_Available"].sum())

print("July OT unavailable:",
      (final_features["OT_July_Available"] == 0).sum())

print("\nOT totals:")
print("June OT:", final_features["Total OT June"].sum())
print("July OT:", final_features["Total OT"].sum())
print("OT Change:", final_features["OT Change"].sum())

print("\nMissing values:")
display(final_features.isna().sum().to_frame("Missing Count"))

FINAL FEATURE VALIDATION
Rows: 20069
Columns: 26

Unique employees: 20069
Duplicate employee codes: 0

June attendance available: 18438
June attendance unavailable: 1631

July OT available: 7970
July OT unavailable: 12099

OT totals:
June OT: 54428.7
July OT: 54714.899999999994
OT Change: 286.2000000000003

Missing values:


,Missing Count
Employee Code,0
Present Count,0
Absent Count,0
Half Day Count,0
Approved Leaves,0
Payable,0
Non Payable,0
Total Days,0
Absence Rate July,0
Present Count June,1631


In [96]:
# ============================================================
# FINAL DATASET VALIDATION
# ============================================================

print("=" * 60)
print("FINAL FEATURE DATASET VALIDATION")
print("=" * 60)

print("Rows:", len(final_features))
print("Columns:", len(final_features.columns))

print("\nUnique employees:",
      final_features["Employee Code"].nunique())

print("Duplicate employee codes:",
      final_features["Employee Code"].duplicated().sum())

print("\nJune attendance available:",
      final_features["June_Attendance_Available"].sum())

print("June attendance unavailable:",
      (final_features["June_Attendance_Available"] == 0).sum())

print("\nJuly OT available:",
      final_features["OT_July_Available"].sum())

print("July OT unavailable:",
      (final_features["OT_July_Available"] == 0).sum())

print("\nOT totals:")
print("June OT:", final_features["Total OT June"].sum())
print("July OT:", final_features["Total OT"].sum())
print("OT Change:", final_features["OT Change"].sum())

print("\nMissing values:")
missing = final_features.isna().sum()
print(missing[missing > 0])

print("\nFinal feature columns:")
print(final_features.columns.tolist())

print("=" * 60)

FINAL FEATURE DATASET VALIDATION
Rows: 20069
Columns: 26

Unique employees: 20069
Duplicate employee codes: 0

June attendance available: 18438
June attendance unavailable: 1631

July OT available: 7970
July OT unavailable: 12099

OT totals:
June OT: 54428.7
July OT: 54714.899999999994
OT Change: 286.2000000000003

Missing values:
Present Count June      1631
Absent Count June       1631
Half Day Count June     1631
Approved Leaves June    1631
Payable June            1631
Non Payable June        1631
Total Days June         1631
Absence Rate June       1631
dtype: int64

Final feature columns:
['Employee Code', 'Present Count', 'Absent Count', 'Half Day Count', 'Approved Leaves', 'Payable', 'Non Payable', 'Total Days', 'Absence Rate July', 'Present Count June', 'Absent Count June', 'Half Day Count June', 'Approved Leaves June', 'Payable June', 'Non Payable June', 'Total Days June', 'Absence Rate June', 'June_Attendance_Available', 'Chargeable OT June', 'Non Chargeable OT June', 'Total

In [97]:
form_data = pd.read_csv(input_file_path / "New Joiners.csv")
print("New Joiners data shape:", form_data.shape)
print("Shape:", form_data.shape)
print("Columns:")
print(form_data.columns.tolist())

display(form_data.head())


New Joiners data shape: (1706, 197)
Shape: (1706, 197)
Columns:
['Employee Status', 'Soft Joining ID', 'Employee Verify Type', 'Onboarding Date', 'Employee Code', 'Employee Name', 'Date Of Joined', 'Date Of Birth', 'Designation', 'Cost Center', 'Gender', 'Marital Status', 'Contact No', 'Emergency No', 'Location', 'Region Name', 'Present Address', 'Permanent Address', 'Site Name', 'Operation Manager', 'Client Verticle', 'Onboarding User', 'PAN Number', 'AADHAR Number', 'ESIC No', 'UAN Number', 'Account No', 'Bank Name', 'IFSC Code', 'Religion', 'Blood Group', 'Nominee Name', 'Relation', 'Fathers Name / Husband Name', 'Higher Education', 'Created Date', 'ABRY BENEFIT', 'ACCOR MEAL VOUCHER', 'ADMIN COST TRAINING & CONTINGENCY COST', 'ANNUAL BONUS', 'ATTENDANCE ALLOWANCE', 'ATTENDANCE INCENTIVE', 'ATTIRE REIMBURSEMENT', 'BASIC', 'BONUS', 'BOOKS AND PERIODICALS', 'CANTEEN DEDUCTION', 'CAR REIMBURSEMENT', 'CASH ALLOWANCE', 'CHILD ALLOWANCE', 'CITY COMPENSATORY ALLOW', 'CO CNTR TO EDLI ADMN C

,Employee Status,Soft Joining ID,Employee Verify Type,Onboarding Date,Employee Code,Employee Name,Date Of Joined,Date Of Birth,Designation,Cost Center,...,UTILITY ALLOWANCE,VEHICAL ALLOWANCE,VEHICLE ALLOWANCE,VERD,VOLUNTARY PF,WASHING ALLOWANCE,WPC,WPC INCENTIVE,YLY LEAVE ENCASHMENT TAXABLE,Monthly Gross Salary
0,NaN,'OF06170,OCR,'26-JUN-26,'583020,KALYANI MANDAL,'26-JUN-2026,'12-MAY-1987,Housekeeper,NSPIRA MANAGEMENT SER PVT LTD TELANGANA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15165.0
1,NaN,'OF06171,OCR,'01-JUN-26,'583163,ROJA V,'01-JUN-2026,'01-JAN-1986,Chambermaid,SHIV NADAR TRUST TAMILNADU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20782.0
2,NaN,'OF06172,OCR,'01-JUN-26,'583164,MEGALA,'01-JUN-2026,'05-APR-1992,Chambermaid,SHIV NADAR TRUST TAMILNADU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20782.0
3,NaN,'OF06173,OCR,'01-JUN-26,'583165,BUVANESWARI S,'01-JUN-2026,'04-AUG-1995,Chambermaid,SHIV NADAR TRUST TAMILNADU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20782.0
4,NaN,'OF06174,OCR,'01-JUN-26,'583166,R DESAPPAN,'01-JUN-2026,'24-NOV-1993,Senior Housekeeper,SHIV NADAR TRUST TAMILNADU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21817.0


In [98]:
# ============================================================
# PREPARE MONTHLY NEW JOINER DATA FOR FORECASTING
# ============================================================


hiring_df = form_data.copy()

print("Hiring dataset shape:", hiring_df.shape)
print("Columns:")
print(hiring_df.columns.tolist())

Hiring dataset shape: (1706, 197)
Columns:
['Employee Status', 'Soft Joining ID', 'Employee Verify Type', 'Onboarding Date', 'Employee Code', 'Employee Name', 'Date Of Joined', 'Date Of Birth', 'Designation', 'Cost Center', 'Gender', 'Marital Status', 'Contact No', 'Emergency No', 'Location', 'Region Name', 'Present Address', 'Permanent Address', 'Site Name', 'Operation Manager', 'Client Verticle', 'Onboarding User', 'PAN Number', 'AADHAR Number', 'ESIC No', 'UAN Number', 'Account No', 'Bank Name', 'IFSC Code', 'Religion', 'Blood Group', 'Nominee Name', 'Relation', 'Fathers Name / Husband Name', 'Higher Education', 'Created Date', 'ABRY BENEFIT', 'ACCOR MEAL VOUCHER', 'ADMIN COST TRAINING & CONTINGENCY COST', 'ANNUAL BONUS', 'ATTENDANCE ALLOWANCE', 'ATTENDANCE INCENTIVE', 'ATTIRE REIMBURSEMENT', 'BASIC', 'BONUS', 'BOOKS AND PERIODICALS', 'CANTEEN DEDUCTION', 'CAR REIMBURSEMENT', 'CASH ALLOWANCE', 'CHILD ALLOWANCE', 'CITY COMPENSATORY ALLOW', 'CO CNTR TO EDLI ADMN CHARGE', 'CO CNTR TO E

In [99]:
# ============================================================
# PREPARE JOINING DATE
# ============================================================

date_col = "Date Of Joined"

hiring_df[date_col] = pd.to_datetime(
    hiring_df[date_col],
    errors="coerce"
)

hiring_df = hiring_df.dropna(subset=[date_col]).copy()

print("Valid joining records:", len(hiring_df))
print(
    "Date range:",
    hiring_df[date_col].min(),
    "to",
    hiring_df[date_col].max()
)

Valid joining records: 1706
Date range: 2026-05-01 00:00:00 to 2026-07-01 00:00:00


C:\Users\veena.sharma\AppData\Local\Temp\ipykernel_25448\4279739609.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hiring_df[date_col] = pd.to_datetime(


In [100]:
# ============================================================
# MONTHLY NEW JOINER COUNTS
# ============================================================

monthly_hiring = (
    hiring_df
    .assign(
        Month=hiring_df[date_col].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("Month")
    .size()
    .reset_index(name="New_Hires")
)

monthly_hiring = monthly_hiring.sort_values("Month").reset_index(drop=True)

print("Monthly hiring data shape:", monthly_hiring.shape)

display(monthly_hiring.head(10))
display(monthly_hiring.tail(10))

Monthly hiring data shape: (3, 2)


,Month,New_Hires
0,2026-05-01,196
1,2026-06-01,1494
2,2026-07-01,16


,Month,New_Hires
0,2026-05-01,196
1,2026-06-01,1494
2,2026-07-01,16


In [101]:
# ============================================================
# PROPHET DATASET
# ============================================================

prophet_df = monthly_hiring.rename(
    columns={
        "Month": "ds",
        "New_Hires": "y"
    }
)

prophet_df = prophet_df[["ds", "y"]].copy()

print("Prophet dataset:")
print(prophet_df.shape)

display(prophet_df)

Prophet dataset:
(3, 2)


,ds,y
0,2026-05-01,196
1,2026-06-01,1494
2,2026-07-01,16


In [102]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_size = int(len(prophet_df) * 0.70)

train_df = prophet_df.iloc[:train_size].copy()
test_df = prophet_df.iloc[train_size:].copy()

print("Total months:", len(prophet_df))
print("Training months:", len(train_df))
print("Testing months:", len(test_df))

print("\nTraining period:")
print(train_df["ds"].min(), "to", train_df["ds"].max())

print("\nTesting period:")
print(test_df["ds"].min(), "to", test_df["ds"].max())

Total months: 3
Training months: 2
Testing months: 1

Training period:
2026-05-01 00:00:00 to 2026-06-01 00:00:00

Testing period:
2026-07-01 00:00:00 to 2026-07-01 00:00:00


In [103]:
!pip install prophet


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [104]:
!pip install matplotlib


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [105]:
%pip install --upgrade prophet matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
# ============================================================
# PROPHET MODEL
# ============================================================

from prophet import Prophet

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.95
)

prophet_model.fit(train_df)

print("Prophet model trained successfully.")

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
20:02:40 - cmdstanpy - INFO - Chain [1] start processing
20:02:40 - cmdstanpy - INFO - Chain [1] done processing


Prophet model trained successfully.


In [110]:
# ============================================================
# MODEL EVALUATION
# ============================================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

forecast_test = prophet_model.predict(test_df[["ds"]])

evaluation_df = test_df.merge(
    forecast_test[["ds", "yhat"]],
    on="ds",
    how="left"
)

mae = mean_absolute_error(
    evaluation_df["y"],
    evaluation_df["yhat"]
)

rmse = np.sqrt(
    mean_squared_error(
        evaluation_df["y"],
        evaluation_df["yhat"]
    )
)

# MAPE excluding zero actual values
non_zero = evaluation_df["y"] != 0

if non_zero.sum() > 0:
    mape = (
        np.mean(
            np.abs(
                (
                    evaluation_df.loc[non_zero, "y"]
                    - evaluation_df.loc[non_zero, "yhat"]
                )
                / evaluation_df.loc[non_zero, "y"]
            )
        ) * 100
    )
else:
    mape = np.nan

print("=" * 60)
print("FUTURE HIRING MODEL EVALUATION")
print("=" * 60)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")

display(evaluation_df)

FUTURE HIRING MODEL EVALUATION
MAE  : 17212.15
RMSE : 17212.15
MAPE : 107575.91%


,ds,y,yhat
0,2026-07-01,16,17228.146276


In [111]:
# ============================================================
# FUTURE HIRING FORECAST
# ============================================================

future_months = 6

future_dates = prophet_model.make_future_dataframe(
    periods=future_months,
    freq="MS"
)

future_forecast = prophet_model.predict(future_dates)

future_hiring_forecast = future_forecast[
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].tail(future_months).copy()

future_hiring_forecast["Forecast New Hires"] = (
    future_hiring_forecast["yhat"]
    .clip(lower=0)
    .round()
    .astype(int)
)

future_hiring_forecast["Lower Bound"] = (
    future_hiring_forecast["yhat_lower"]
    .clip(lower=0)
    .round()
    .astype(int)
)

future_hiring_forecast["Upper Bound"] = (
    future_hiring_forecast["yhat_upper"]
    .clip(lower=0)
    .round()
    .astype(int)
)

future_hiring_forecast = future_hiring_forecast[
    [
        "ds",
        "Forecast New Hires",
        "Lower Bound",
        "Upper Bound"
    ]
]

future_hiring_forecast = future_hiring_forecast.rename(
    columns={"ds": "Forecast Month"}
)

print("=" * 60)
print("FUTURE HIRING NEEDS FORECAST")
print("=" * 60)

display(future_hiring_forecast)

FUTURE HIRING NEEDS FORECAST


,Forecast Month,Forecast New Hires,Lower Bound,Upper Bound
2,2026-07-01,17228,17228,17228
3,2026-08-01,11783,11783,11783
4,2026-09-01,13944,13944,13944
5,2026-10-01,8278,8278,8278
6,2026-11-01,26905,26905,26905
7,2026-12-01,41789,41789,41789


In [112]:
# ============================================================
# FINAL FUTURE HIRING MODEL SUMMARY
# ============================================================

print("=" * 70)
print("FINAL FUTURE HIRING NEEDS FORECAST MODEL")
print("=" * 70)

print(f"Historical months available : {len(prophet_df)}")
print(f"Training months             : {len(train_df)}")
print(f"Testing months              : {len(test_df)}")

print(f"\nMAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")

print("\nFuture hiring forecast:")
display(future_hiring_forecast)

print("\nModel: Prophet Time-Series Forecasting")
print("Target: Monthly New Hires")
print("Forecast horizon: 6 months")


FINAL FUTURE HIRING NEEDS FORECAST MODEL
Historical months available : 3
Training months             : 2
Testing months              : 1

MAE  : 17212.15
RMSE : 17212.15
MAPE : 107575.91%

Future hiring forecast:


,Forecast Month,Forecast New Hires,Lower Bound,Upper Bound
2,2026-07-01,17228,17228,17228
3,2026-08-01,11783,11783,11783
4,2026-09-01,13944,13944,13944
5,2026-10-01,8278,8278,8278
6,2026-11-01,26905,26905,26905
7,2026-12-01,41789,41789,41789



Model: Prophet Time-Series Forecasting
Target: Monthly New Hires
Forecast horizon: 6 months


In [113]:
# ============================================================
# SAVE FINAL MODEL OUTPUTS
# ============================================================

final_hiring_output = future_hiring_forecast.copy()

final_hiring_output.to_excel(
    "Future_Hiring_Needs_Forecast.xlsx",
    index=False
)

print("Saved: Future_Hiring_Needs_Forecast.xlsx")

Saved: Future_Hiring_Needs_Forecast.xlsx


In [114]:
# ============================================================
# FINAL FUTURE HIRING FORECAST WITH PREDICTIVE FACTORS
# ============================================================

# Generate next 6 months
future = prophet_model.make_future_dataframe(
    periods=6,
    freq="MS"
)

forecast = prophet_model.predict(future)

# Keep only future months
future_forecast = forecast[
    forecast["ds"] > train_df["ds"].max()
].copy()

# Prepare final forecast table
final_forecast = future_forecast[
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].copy()

# Rename columns
final_forecast.columns = [
    "Forecast Month",
    "Forecast New Hires",
    "Lower Bound",
    "Upper Bound"
]

# Clean predicted values
final_forecast["Forecast New Hires"] = (
    final_forecast["Forecast New Hires"]
    .clip(lower=0)
    .round()
    .astype(int)
)

final_forecast["Lower Bound"] = (
    final_forecast["Lower Bound"]
    .clip(lower=0)
    .round()
    .astype(int)
)

final_forecast["Upper Bound"] = (
    final_forecast["Upper Bound"]
    .clip(lower=0)
    .round()
    .astype(int)
)

# Display
display(final_forecast)

,Forecast Month,Forecast New Hires,Lower Bound,Upper Bound
2,2026-07-01,17228,17228,17228
3,2026-08-01,11783,11783,11783
4,2026-09-01,13944,13944,13944
5,2026-10-01,8278,8278,8278
6,2026-11-01,26905,26905,26905
7,2026-12-01,41789,41789,41789


In [115]:
# ============================================================
# EXPORT FUTURE HIRING FORECAST TO EXCEL
# ============================================================

output_file = "Future_Hiring_Forecast.xlsx"

final_forecast.to_excel(
    output_file,
    index=False
)

print("Excel file created successfully:")
print(output_file)

Excel file created successfully:
Future_Hiring_Forecast.xlsx


In [117]:
# ============================================================
# FIND MONTHLY FACTOR DATAFRAME
# ============================================================

import pandas as pd

required_factors = {
    "absenteeism_rate",
    "average_tenure_years",
    "avg_ot_hours_per_emp",
    "budget",
    "headcount",
    "resigned_employees",
    "unplanned_leave_days"
}

candidates = []

# Take a snapshot first
all_variables = list(globals().items())

for name, obj in all_variables:

    if isinstance(obj, pd.DataFrame):

        columns = set(obj.columns)
        matched = required_factors.intersection(columns)

        if len(matched) >= 3:
            candidates.append(
                {
                    "name": name,
                    "shape": obj.shape,
                    "matched": sorted(matched),
                    "columns": list(obj.columns)
                }
            )

print("=" * 70)
print("POSSIBLE MONTHLY FACTOR DATAFRAMES")
print("=" * 70)

if candidates:

    for item in candidates:
        print("\nDataFrame name :", item["name"])
        print("Shape          :", item["shape"])
        print("Matching factors:")
        print(item["matched"])

else:

    print("\nNo suitable monthly dataframe found.")

    print("\nAvailable DataFrames:")

    for name, obj in all_variables:
        if isinstance(obj, pd.DataFrame):
            print(f"  {name}: {obj.shape}")

POSSIBLE MONTHLY FACTOR DATAFRAMES

No suitable monthly dataframe found.

Available DataFrames:
  employee_master: (20486, 93)
  yearly_joiners: (23, 2)
  monthly_hiring: (3, 2)
  monthly_headcount: (266, 2)
  workforce_data: (266, 5)
  resignation_data: (9, 2)
  monthly_tenure: (266, 2)
  active: (20222, 93)
  temp: (4783, 112)
  june_attendance: (19569, 69)
  july_attendance: (20069, 72)
  attendance_metrics: (2, 3)
  july_ot: (4783, 112)
  july_ot_feature: (4781, 4)
  june_ot: (3228, 109)
  june_ot_feature: (3218, 4)
  ot_features: (7999, 8)
  june_att_feature: (19569, 9)
  july_att_feature: (20069, 11)
  df: (7999, 8)
  final_features: (20069, 26)
  june_att_merge: (19569, 9)
  june_merge: (19569, 9)
  ot_merge: (7999, 7)
  form_data: (1706, 197)
  hiring_df: (1706, 197)
  prophet_df: (3, 2)
  train_df: (2, 2)
  test_df: (1, 2)
  forecast_test: (1, 16)
  evaluation_df: (1, 3)
  future_dates: (8, 1)
  future_forecast: (6, 16)
  future_hiring_forecast: (6, 4)
  final_hiring_output: (

In [118]:
# ============================================================
# CHECK EXISTING MONTHLY DATA COLUMNS
# ============================================================

check_dfs = [
    "monthly_hiring",
    "monthly_headcount",
    "workforce_data",
    "monthly_tenure",
    "resignation_data",
    "attendance_metrics"
]

for name in check_dfs:
    obj = globals().get(name)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    if obj is not None:
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
        print("\nSample:")
        display(obj.head(3))


monthly_hiring
Shape: (3, 2)
Columns: ['Month', 'New_Hires']

Sample:


,Month,New_Hires
0,2026-05-01,196
1,2026-06-01,1494
2,2026-07-01,16



monthly_headcount
Shape: (266, 2)
Columns: ['Month', 'Headcount']

Sample:


,Month,Headcount
0,2004-06-01,1
1,2004-07-01,1
2,2004-08-01,1



workforce_data
Shape: (266, 5)
Columns: ['Month', 'New Joiners', 'Headcount', 'Resigned Employees', 'Turnover Rate (%)']

Sample:


...

,Month,Average Tenure Years
0,2004-06-01,0.079398
1,2004-07-01,0.164271
2,2004-08-01,0.249144



resignation_data
Shape: (9, 2)
Columns: ['Month', 'Resigned Employees']

Sample:


,Month,Resigned Employees
0,2025-12-01,1
1,2026-01-01,1
2,2026-02-01,3



attendance_metrics
Shape: (2, 3)
Columns: ['Month', 'Absenteeism Rate (%)', 'Unplanned Leave Days']

Sample:


,Month,Absenteeism Rate (%),Unplanned Leave Days
0,2026-06-01,6.370338,35175.5
1,2026-07-01,7.378252,43841.0


In [ ]:
# ============================================================
# FINAL FUTURE HIRING OUTPUT
# PREDICTION + PREDICTIVE FACTORS
# ============================================================

# Copy forecast
output_df = final_forecast.copy()

# Rename forecast date for merging
output_df["Month"] = pd.to_datetime(output_df["Forecast Month"])

# ------------------------------------------------------------
# IMPORTANT:
# Replace monthly_factors below ONLY if your existing
# monthly factor dataframe has a different variable name.
# ------------------------------------------------------------

monthly_factors = monthly_factors.copy()

monthly_factors["Month"] = pd.to_datetime(monthly_factors["Month"])

# ------------------------------------------------------------
# Predictive factors required in final output
# ------------------------------------------------------------

factor_columns = [
    "absenteeism_rate",
    "average_tenure_years",
    "avg_ot_hours_per_emp",
    "budget",
    "headcount",
    "resigned_employees",
    "unplanned_leave_days"
]

# Check which factors are actually available
available_factors = [
    col for col in factor_columns
    if col in monthly_factors.columns
]

missing_factors = [
    col for col in factor_columns
    if col not in monthly_factors.columns
]

print("Available predictive factors:")
print(available_factors)

if missing_factors:
    print("\nWARNING - Missing factor columns:")
    print(missing_factors)

# ------------------------------------------------------------
# Merge factors with forecast
# ------------------------------------------------------------

output_df = output_df.merge(
    monthly_factors[["Month"] + available_factors],
    on="Month",
    how="left"
)

# ------------------------------------------------------------
# Final column
# ------------------------------------------------------------

output_df["Predicted Hiring"] = output_df["Forecast New Hires"]

# Keep required columns
final_output_columns = (
    ["Month"]
    + available_factors
    + ["Predicted Hiring"]
)

final_output = output_df[final_output_columns].copy()

# Round prediction
final_output["Predicted Hiring"] = (
    final_output["Predicted Hiring"]
    .clip(lower=0)
    .round()
    .astype(int)
)

# Sort by month
final_output = final_output.sort_values("Month").reset_index(drop=True)

# Display
display(final_output)